# Session 2 · Object-Oriented Programming Continued

Run all cells in order with a Python 3 kernel. Only the standard library is required. File exercises use temporary directories.

Sources: [in-class notebook](https://github.com/ReDI-Aarhus/Agentic-AI/blob/main/classes/02-oop-continued/code/class2_exercises.ipynb), [homework notebook](https://github.com/ReDI-Aarhus/Agentic-AI/blob/main/classes/02-oop-continued/homework/class2_homework.ipynb), and the Session 2 lecture slides.

In [1]:
import csv
from pathlib import Path
import re
from tempfile import TemporaryDirectory

## Warm-up · Task lists and a simple agent

The slides revisit Session 1 B3–B4. This example picks the highest-priority pending task, marks it done, and leaves lower-priority tasks for subsequent calls.

In [2]:
class Task:
    def __init__(self, name, priority):
        self.name = name
        self.priority = priority
        self.done = False

    def mark_done(self):
        self.done = True


class TaskList:
    def __init__(self):
        self.tasks = []

    def add(self, task):
        self.tasks.append(task)

    def __len__(self):
        return len(self.tasks)

    def pending(self):
        return sorted(
            (task for task in self.tasks if not task.done),
            key=lambda task: task.priority,
        )


class SimpleAgent:
    def __init__(self, name, task_list):
        self.name = name
        self.task_list = task_list

    def act(self):
        pending = self.task_list.pending()
        if not pending:
            return f"{self.name}: nothing to do!"
        task = pending[0]
        task.mark_done()
        return f"{self.name}: working on '{task.name}'"


task_list = TaskList()
task_list.add(Task("read", 2))
task_list.add(Task("code", 1))
agent = SimpleAgent("Helper", task_list)
assert len(task_list) == 2
assert [agent.act() for _ in range(3)] == [
    "Helper: working on 'code'",
    "Helper: working on 'read'",
    "Helper: nothing to do!",
]
print("B3–B4 review passed.")

B3–B4 review passed.


## Coffee Shop · Composition, comparisons, and overriding

Orders compare by listed drink price. The delivery total adds the delivery fee from the first extension and the optional tip from the revenue exercise. The shop calls each order's own `get_total_price()`.

In [3]:
class CoffeeOrder:
    def __init__(self, customer_name, drink_name, size, price):
        size = size.strip().lower()
        if size not in {"small", "medium", "large"}:
            raise ValueError("Size must be small, medium, or large.")
        self.customer_name = customer_name
        self.drink_name = drink_name
        self.size = size
        self.price = price

    def get_order_summary(self):
        return f"{self.customer_name} ordered a {self.size} {self.drink_name} for {self.price:g} dkk"

    def get_total_price(self):
        return self.price

    def __lt__(self, other):
        if not isinstance(other, CoffeeOrder):
            return NotImplemented
        return self.price < other.price


class CoffeeShop:
    def __init__(self, shop_name):
        self.shop_name = shop_name
        self.orders = []

    def add_order(self, order):
        self.orders.append(order)

    def get_total_orders(self):
        return len(self.orders)

    def list_all_orders(self):
        for order in self.orders:
            print(order.get_order_summary())

    def get_revenue(self):
        return sum(order.get_total_price() for order in self.orders)


class DineInCoffeeOrder(CoffeeOrder):
    def __init__(self, customer_name, drink_name, size, price, table_number):
        super().__init__(customer_name, drink_name, size, price)
        self.table_number = table_number

    def get_total_price(self):
        return super().get_total_price()


class DeliveryCoffeeOrder(CoffeeOrder):
    def __init__(self, customer_name, drink_name, size, price, delivery_address, delivery_fee, tip=0):
        super().__init__(customer_name, drink_name, size, price)
        self.delivery_address = delivery_address
        self.delivery_fee = delivery_fee
        self.tip = tip

    def get_total_price(self):
        return super().get_total_price() + self.delivery_fee + self.tip

In [4]:
takeaway = CoffeeOrder("Alice", "latte", "Large", 20)
dine_in = DineInCoffeeOrder("Sam", "cappuccino", "medium", 35, 4)
delivery = DeliveryCoffeeOrder("Jo", "flat white", "small", 40, "Example address", 15)
tipped = DeliveryCoffeeOrder("Lee", "mocha", "large", 42, "Example address", 15, tip=5)
shop = CoffeeShop("Practice Coffee Shop")
for order in (takeaway, dine_in, delivery, tipped):
    shop.add_order(order)

assert takeaway.get_order_summary() == "Alice ordered a large latte for 20 dkk"
assert takeaway < dine_in < delivery
assert shop.get_total_orders() == 4
assert dine_in.table_number == 4 and dine_in.get_total_price() == 35
assert delivery.get_total_price() == 55 and tipped.get_total_price() == 62
assert shop.get_revenue() == 172
try:
    CoffeeOrder("Alice", "latte", "extra large", 20)
except ValueError:
    pass
else:
    raise AssertionError("Invalid sizes must be rejected.")

shop.list_all_orders()
print(f"Total revenue: {shop.get_revenue()} dkk")

Alice ordered a large latte for 20 dkk
Sam ordered a medium cappuccino for 35 dkk
Jo ordered a small flat white for 40 dkk
Lee ordered a large mocha for 42 dkk
Total revenue: 172 dkk


## Library exercises 1–2 · Book and EBook

`Book` uses a private price and protected stock. Comparisons use price, and the subclass extends the string representation with its file size.

In [5]:
class Book:
    def __init__(self, title, author, price, stock):
        self.title = title
        self.author = author
        self.__price = price
        self._stock = stock

    def get_price(self):
        return self.__price

    def get_stock(self):
        return self._stock

    def sell(self):
        if self._stock <= 0:
            return False
        self._stock -= 1
        return True

    def __str__(self):
        return f"{self.title} by {self.author} - {self.__price:g} dkk ({self._stock} in stock)"

    def __eq__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return self.get_price() == other.get_price()

    def __lt__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return self.get_price() < other.get_price()


class EBook(Book):
    def __init__(self, title, author, price, stock, file_size_mb):
        super().__init__(title, author, price, stock)
        self.file_size_mb = file_size_mb

    def __str__(self):
        return f"{super().__str__()} [ebook, {self.file_size_mb:g}MB]"


dune = Book("Dune", "Frank Herbert", 150, 3)
ebook = EBook("Stories, Volume 1", "Frank Herbert", 75.50, 1, 4.2)
assert str(dune) == "Dune by Frank Herbert - 150 dkk (3 in stock)"
assert dune.sell() and dune.get_stock() == 2
assert not Book("Sold out", "Author", 50, 0).sell()
assert ebook < dune
assert Book("Other title", "Another author", 150, 1) == dune
assert str(ebook).endswith(" [ebook, 4.2MB]")
print(dune)
print(ebook)

Dune by Frank Herbert - 150 dkk (2 in stock)
Stories, Volume 1 by Frank Herbert - 75.5 dkk (1 in stock) [ebook, 4.2MB]


## Library exercises 3–6 · Catalog, iteration, CSV, and merging

The catalog CSV stores title, author, price, and stock. Loading reconstructs plain books. Merging creates a new list that holds the original book objects.

In [6]:
class Library:
    def __init__(self, name):
        self.name = name
        self.books = []
        self._index = 0

    def add_book(self, book):
        self.books.append(book)

    def total_value(self):
        return sum(book.get_price() * book.get_stock() for book in self.books)

    def find_by_author(self, author):
        return [book for book in self.books if book.author == author]

    def __len__(self):
        return len(self.books)

    def __getitem__(self, index):
        return self.books[index]

    def __iter__(self):
        self._index = 0
        return self

    def __next__(self):
        if self._index >= len(self.books):
            raise StopIteration
        book = self.books[self._index]
        self._index += 1
        return book

    def save_to_file(self, filename):
        with open(filename, "w", encoding="utf-8", newline="") as file:
            writer = csv.writer(file)
            for book in self.books:
                writer.writerow((book.title, book.author, book.get_price(), book.get_stock()))

    @classmethod
    def load_from_file(cls, filename, name):
        catalog = cls(name)
        with open(filename, encoding="utf-8", newline="") as file:
            for title, author, price, stock in csv.reader(file):
                catalog.add_book(Book(title, author, float(price), int(stock)))
        return catalog

    def __add__(self, other):
        if not isinstance(other, Library):
            return NotImplemented
        merged = Library(f"{self.name} + {other.name}")
        merged.books = self.books + other.books
        return merged

In [7]:
library = Library("Central")
library.add_book(dune)
library.add_book(ebook)
assert len(library) == 2 and library[0] is dune
assert library.find_by_author("Frank Herbert") == [dune, ebook]
assert library.find_by_author("Unknown") == []
assert library.total_value() == 375.50
assert [book.title for book in library] == ["Dune", "Stories, Volume 1"]
assert [book.title for book in library] == ["Dune", "Stories, Volume 1"]

with TemporaryDirectory() as folder:
    catalog = Path(folder) / "catalog.csv"
    library.save_to_file(catalog)
    restored = Library.load_from_file(catalog, "Restored")
    assert type(restored[1]) is Book
    assert restored[1].title == ebook.title and restored[1].author == ebook.author
    assert restored.total_value() == library.total_value()

branch = Library("Branch")
branch.add_book(Book("New arrival", "Author", 25, 4))
merged = library + branch
assert len(merged) == 3
assert len(library) == 2 and len(branch) == 1
assert merged.total_value() == 475.50
print(f"Library checks passed. Merged stock value: {merged.total_value():.2f} dkk.")

Library checks passed. Merged stock value: 475.50 dkk.


## Lecture file exercises · Open modes and text processing

The first cell checks read, write, append, and exclusive creation. The second covers all eight reading and writing tasks.

In [8]:
sample_content = "Hello, World!\nThis is line 2.\nThis is line 3.\n"

with TemporaryDirectory() as folder:
    root = Path(folder)
    sample = root / "sample.txt"
    sample.write_text(sample_content, encoding="utf-8")
    with sample.open("r", encoding="utf-8"):
        pass
    assert sample.read_text(encoding="utf-8") == sample_content

    with sample.open("w", encoding="utf-8"):
        pass
    assert sample.read_text(encoding="utf-8") == ""
    print("Write mode truncated sample.txt.")

    sample.write_text(sample_content, encoding="utf-8")
    with sample.open("a", encoding="utf-8"):
        pass
    assert sample.read_text(encoding="utf-8") == sample_content

    try:
        with (root / "nonexistent.txt").open("r", encoding="utf-8"):
            pass
    except FileNotFoundError:
        print("Read mode raised FileNotFoundError for a missing file.")
    else:
        raise AssertionError("Read mode must fail for a missing file.")

    new_file = root / "newfile.txt"
    with new_file.open("x", encoding="utf-8"):
        pass
    try:
        with new_file.open("x", encoding="utf-8"):
            pass
    except FileExistsError:
        print("Exclusive creation raised FileExistsError on the second attempt.")
    else:
        raise AssertionError("Exclusive creation must reject an existing file.")

Write mode truncated sample.txt.
Read mode raised FileNotFoundError for a missing file.
Exclusive creation raised FileExistsError on the second attempt.


In [9]:
poem = "Roses are red,\nViolets are blue,\nPython is awesome,\nAnd so are you!\n"
items = ["notebook", "pencil", "mug", "book", "lamp"]

with TemporaryDirectory() as folder:
    root = Path(folder)
    source = root / "input.txt"
    source.write_text(poem, encoding="utf-8")

    with source.open(encoding="utf-8") as file:
        content = file.read()
        print(content, end="")

    with source.open(encoding="utf-8") as file:
        for number, line in enumerate(file, start=1):
            print(f"{number}. {line.rstrip()}")

    output = root / "output.txt"
    with output.open("w", encoding="utf-8") as file:
        file.write("Line 1\nLine 2\nLine 3\n")

    with source.open(encoding="utf-8") as file:
        lines = file.readlines()
    with (root / "copy.txt").open("w", encoding="utf-8") as file:
        file.writelines(lines)

    (root / "uppercase.txt").write_text(content.upper(), encoding="utf-8")
    with (root / "items.txt").open("w", encoding="utf-8") as file:
        file.writelines(f"{item}\n" for item in items)

    with output.open("a", encoding="utf-8") as file:
        file.write("This is a new line!\n")

    with source.open(encoding="utf-8") as original, (root / "filtered.txt").open(
        "w", encoding="utf-8"
    ) as filtered:
        filtered.writelines(line for line in original if re.search(r"\bare\b", line))

    assert (root / "copy.txt").read_text(encoding="utf-8") == poem
    assert (root / "uppercase.txt").read_text(encoding="utf-8") == poem.upper()
    assert (root / "items.txt").read_text(encoding="utf-8").splitlines() == items
    assert output.read_text(encoding="utf-8") == "Line 1\nLine 2\nLine 3\nThis is a new line!\n"
    assert (root / "filtered.txt").read_text(encoding="utf-8").splitlines() == [
        "Roses are red,", "Violets are blue,", "And so are you!"
    ]

print("Text processing checks passed.")

Roses are red,
Violets are blue,
Python is awesome,
And so are you!
1. Roses are red,
2. Violets are blue,
3. Python is awesome,
4. And so are you!
Text processing checks passed.


## Homework 1–2 · Membership classes

Member IDs come from a shared counter. Equality uses ID, and ordering uses age. The basic fee is 200 DKK. Premium costs 500 DKK plus 150 DKK when a trainer is assigned.

In [10]:
class Member:
    _id_counter = 0

    def __init__(self, name, email, age):
        self.name = name
        self.email = email
        self._age = age
        Member._id_counter += 1
        self.__member_id = Member._id_counter

    def get_member_id(self):
        return self.__member_id

    def get_age(self):
        return self._age

    def __str__(self):
        return f"Member #{self.__member_id}: {self.name} ({self.email})"

    def __eq__(self, other):
        if not isinstance(other, Member):
            return NotImplemented
        return self.get_member_id() == other.get_member_id()

    def __lt__(self, other):
        if not isinstance(other, Member):
            return NotImplemented
        return self.get_age() < other.get_age()


class BasicMember(Member):
    def __init__(self, name, email, age):
        super().__init__(name, email, age)

    def get_monthly_fee(self):
        return 200


class PremiumMember(Member):
    def __init__(self, name, email, age, personal_trainer=None):
        super().__init__(name, email, age)
        self.personal_trainer = personal_trainer

    def get_monthly_fee(self):
        return 500 + (150 if self.personal_trainer else 0)

## Homework 3–5 · Gym, iteration, and member CSV

The assignment's CSV contains name, email, and age. Reloading assigns new IDs to plain `Member` objects. A plain member has no plan, so it contributes zero to monthly revenue.

In [11]:
class Gym:
    def __init__(self, name):
        self.name = name
        self.members = []
        self._index = 0

    def add_member(self, member):
        self.members.append(member)

    def total_monthly_revenue(self):
        return sum(
            member.get_monthly_fee()
            for member in self.members
            if isinstance(member, (BasicMember, PremiumMember))
        )

    def find_by_email(self, email):
        return next((member for member in self.members if member.email == email), None)

    def __len__(self):
        return len(self.members)

    def __getitem__(self, index):
        return self.members[index]

    def __iter__(self):
        self._index = 0
        return self

    def __next__(self):
        if self._index >= len(self.members):
            raise StopIteration
        member = self.members[self._index]
        self._index += 1
        return member

    def __bool__(self):
        return len(self) > 0

    def save_to_file(self, filename):
        with open(filename, "w", encoding="utf-8", newline="") as file:
            writer = csv.writer(file)
            for member in self.members:
                writer.writerow((member.name, member.email, member.get_age()))

    @classmethod
    def load_from_file(cls, filename, name):
        gym = cls(name)
        with open(filename, encoding="utf-8", newline="") as file:
            for member_name, email, age in csv.reader(file):
                gym.add_member(Member(member_name, email, int(age)))
        return gym

In [12]:
start_id = Member._id_counter
basic = BasicMember("Ada Example", "ada@example.com", 24)
premium = PremiumMember("Sam Example", "sam@example.com", 31)
coached = PremiumMember("Lee, Example", "lee@example.com", 29, "Coach Kim")
assert [member.get_member_id() for member in (basic, premium, coached)] == [
    start_id + 1, start_id + 2, start_id + 3
]
assert str(basic) == f"Member #{start_id + 1}: Ada Example (ada@example.com)"
assert basic == basic and basic != coached
assert sorted([premium, basic, coached]) == [basic, coached, premium]
assert [member.get_monthly_fee() for member in (basic, premium, coached)] == [200, 500, 650]

gym = Gym("Practice Gym")
assert not gym and gym.total_monthly_revenue() == 0
for member in (basic, premium, coached):
    gym.add_member(member)

assert gym and len(gym) == 3 and gym[-1] is coached
assert gym.find_by_email("sam@example.com") is premium
assert gym.find_by_email("missing@example.com") is None
assert gym.total_monthly_revenue() == 1350
assert [member.name for member in gym] == ["Ada Example", "Sam Example", "Lee, Example"]
assert list(gym) == [basic, premium, coached]

with TemporaryDirectory() as folder:
    filename = Path(folder) / "members.csv"
    gym.save_to_file(filename)
    loaded = Gym.load_from_file(filename, "Restored Gym")
    assert all(type(member) is Member for member in loaded)
    assert [(member.name, member.email, member.get_age()) for member in loaded] == [
        (member.name, member.email, member.get_age()) for member in gym
    ]
    assert loaded[0].get_member_id() == coached.get_member_id() + 1
    assert loaded.total_monthly_revenue() == 0

print(f"Gym homework checks passed. Monthly revenue: {gym.total_monthly_revenue()} dkk.")

Gym homework checks passed. Monthly revenue: 1350 dkk.
